# Actividad 2 · Estructura, diccionario y validación inicial

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 1 pomodoro (20 minutos)

> Realizá solamente uno de los dos recorridos equivalentes disponibles: **SQL + Python** o **SQL + R**.

En esta actividad vas a inspeccionar la estructura de la tabla y contrastar algunos valores con el diccionario de datos antes de interpretarlos.


## Preparación automática

Ejecutá la celda siguiente sin modificarla. Preparará los datos y la conexión con DuckDB. No necesitás descargar ni editar ningún archivo auxiliar.


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AllenDowney/ThinkStats/v3/data/2002FemPreg.dat.gz"
DATA_PATH = Path("2002FemPreg.dat.gz")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

colspecs = [
    (0, 12), (12, 14), (56, 58), (58, 60), (274, 276),
    (276, 277), (277, 279), (283, 287), (422, 440),
]
names = [
    "caseid", "pregordr", "birthwgt_lb", "birthwgt_oz", "prglngth",
    "outcome", "birthord", "agepreg", "finalwgt",
]

embarazos = pd.read_fwf(
    DATA_PATH, names=names, colspecs=colspecs, compression="gzip"
)
embarazos["birthwgt_lb"] = embarazos["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)
embarazos["birthwgt_oz"] = embarazos["birthwgt_oz"].replace([97, 98, 99], np.nan)
embarazos["totalwgt_lb"] = embarazos["birthwgt_lb"] + embarazos["birthwgt_oz"] / 16
embarazos["totalwgt_kg"] = (embarazos["totalwgt_lb"] * 0.45359237).round(3)
embarazos["agepreg"] = embarazos["agepreg"] / 100
embarazos = embarazos[[
    "caseid", "pregordr", "prglngth", "outcome", "birthord",
    "birthwgt_lb", "birthwgt_oz", "totalwgt_lb", "totalwgt_kg",
    "agepreg", "finalwgt",
]]

con = duckdb.connect(":memory:")
con.register("embarazos_df", embarazos)
con.execute("CREATE OR REPLACE TABLE embarazos AS SELECT * FROM embarazos_df")
print(f"Entorno preparado: {len(embarazos):,} registros disponibles en la tabla 'embarazos'.")


## Estructura de la tabla

Comencemos por identificar las columnas y sus tipos de datos.


In [ ]:
con.sql("""
DESCRIBE embarazos
""")


Observemos también la cantidad de registros y las primeras filas.


In [ ]:
con.sql("""
SELECT COUNT(*) AS total_filas FROM embarazos
""")


In [ ]:
con.sql("""
SELECT * FROM embarazos LIMIT 10
""")


## Validación con el diccionario de datos

Un **diccionario de datos** o *codebook* explica qué representa cada variable y cómo se codifican sus valores. Para interpretar correctamente `outcome`, contrastaremos la tabla con la documentación del NSFG.

| Código | Significado | Cantidad esperada |
|---:|---|---:|
| 1 | Nacimiento vivo | 9.148 |
| 2 | Aborto inducido | 1.862 |
| 3 | Mortinato | 120 |
| 4 | Aborto espontáneo | 1.921 |
| 5 | Embarazo ectópico | 190 |
| 6 | Embarazo en curso | 352 |


In [ ]:
con.sql("""
SELECT outcome, COUNT(*) AS cantidad
FROM embarazos
GROUP BY outcome
ORDER BY outcome
""")


La coincidencia con el diccionario permite comprobar que la variable se cargó correctamente. Esto no demuestra que todo el dataset sea perfecto, pero constituye una validación inicial necesaria.

Revisemos ahora los datos disponibles de peso al nacer.


In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total,
    COUNT(birthwgt_lb) AS con_peso,
    COUNT(*) - COUNT(birthwgt_lb) AS sin_peso
FROM embarazos
""")


Los valores ausentes no significan siempre “error”: en embarazos que no finalizaron con un nacimiento vivo, el peso al nacer no corresponde. El contexto y la definición de la variable son necesarios para decidir cómo interpretarlos.

La columna `totalwgt_kg` fue preparada automáticamente a partir del peso informado en libras y onzas. Comprobemos su promedio para los nacimientos vivos.


In [ ]:
con.sql("""
SELECT
    ROUND(AVG(totalwgt_kg), 3) AS peso_promedio_kg,
    COUNT(totalwgt_kg) AS casos_con_peso
FROM embarazos
WHERE outcome = 1
""")


## Ejercicio

Escribí una consulta SQL que cuente cuántos registros hay para cada valor de `birthord`. Ordená el resultado de menor a mayor. Antes de ejecutarla, anticipá qué debería ocurrir con los valores nulos.


In [ ]:
# Escribí tu consulta SQL acá


## Cierre

Una exploración confiable comienza por reconocer la estructura, consultar el diccionario y validar códigos, rangos y ausencias. Todavía no estamos explicando el fenómeno: estamos verificando qué datos tenemos y cómo deben leerse.

**Bibliografía:** Downey, A. B. (2025). *Think Stats* (3.ª ed.), capítulo 1. [Codebook del NSFG, archivo de embarazos](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NSFG/Cycle6Codebook-Pregnancy.pdf).


In [ ]:
con.close()
